# Historical reproduction and related-repository audit

Preserve the audit trail without mixing obsolete results into the corrected scientific conclusions.

**Mode:** executed analysis of the committed corrected results. Expensive model refitting is available through Notebook 11 and `scripts/reproduce.py`. Replaying saved results is not presented as fresh model training.

In [1]:
from pathlib import Path
import sys, json, itertools
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.revision-repository').exists())
sys.path.insert(0, str(ROOT / 'analysis'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from notebook_support import *
from analysis import PROTEINS, TARGET, FIXED, global_quantile, local_intervals
pd.set_option('display.max_columns', 14)
pd.set_option('display.max_rows', 30)
pd.set_option('display.precision', 5)


## Exact historical reproduction

Historical runs use the original mismatched public matrix solely to explain the submitted numbers and exclusions. They are not revised biological evidence. The original test omitted ID 1298; the reinstated evaluation keeps the same point model.

In [2]:
for name in ['original_reproduction','original_reinstated']:
    cfg=read_json(ROOT/f'archive/historical/runs/{name}/complete.json')
    display(pd.Series({k:cfg[k] for k in ['run','counts','rmse']}).to_frame('Historical result'))
    if 'reproduction' in cfg: display(pd.Series(cfg['reproduction']).to_frame('Difference from submitted outputs'))

,Historical result
run,original_reproduction
counts,"{'train': 8096, 'calibration': 2024, 'test': 2..."
rmse,0.50057


,Difference from submitted outputs
max_prediction_difference,8.88178e-16
max_joint_width_difference,4.44089e-16
covered_count_difference,0.00000e+00


,Historical result
run,original_reinstated
counts,"{'train': 8096, 'calibration': 2024, 'test': 2..."
rmse,0.50158


,Difference from submitted outputs
max_prediction_difference,8.88178e-16
max_joint_width_difference,4.44089e-16
covered_count_difference,0.00000e+00


## Related repository: do downstream notebooks repair the join?

At Toxicodynamics-aware-computational-toxicology commit 5a3b2a2, consumers load df_final.csv and select columns; no downstream SMILES-keyed ADMET reconstruction occurs. Its main saved train/val/test split is nevertheless cluster-disjoint. Do not transfer the historical Beyond-RMSE cluster-overlap counts to it.

In [3]:
audit=read_json(ROOT/'provenance/related_repository_audit_results.json')
display(pd.Series(audit['saved_splits']['sizes'],name='Molecules').to_frame())
display(pd.DataFrame(audit['saved_splits']['pairs']))
consumer=read_json(ROOT/'provenance/related_repository_consumer_checks.json')
display(pd.DataFrame(consumer['features']).T)
display(pd.DataFrame(consumer['wrong_associations_by_split']).T)

,Molecules
train,8043
val,2071
test,2537


,pair,shared_ids,shared_clusters
0,train/val,0,0
1,train/test,0,0
2,val/test,0,0


,n_features,admet_columns,smiles_columns
baseline,2094,0,[]
pca,2094,0,[]
adme,2162,68,[]
plain,2050,0,[]
bbb_pass,2094,0,[]


,n,wrong_ADMET
train,8043,1102
val,2071,259
test,2537,317


## Independent source-version differences

The related repository clips positive docking scores to zero. Those values differ from the historical source retained in this article reanalysis. Its README, summary.csv and experiment_config describe inconsistent result versions. The corrected article outputs here must not be presented as an exact rerun of that other repository.

Master-table MW and ADMET MW also have distinct provenance. The global models and the fixed confidence predictor use their documented respective definitions. No new docking or external ADMET API calculation is claimed.

In [4]:
p=ROOT/'archive/historical/molecular_weight_audit_summary.json'
if p.exists():display(read_json(p))
print('All imported source bytes are traced in provenance/import_manifest.json.')

{'MW_vs_cleaned_rdkit_MolWt': {'mean_abs_difference': 12.215047743261438,
  'max_abs_difference': 1095.5929999999998,
  'fraction_within_0_01': 7.904513477195479e-05},
 'MW_vs_cleaned_rdkit_ExactMolWt': {'mean_abs_difference': 11.927872161842465,
  'max_abs_difference': 1091.9204798199994,
  'fraction_within_0_01': 0.8422259109951783},
 'MW_g_per_mol_vs_cleaned_rdkit_MolWt': {'mean_abs_difference': 0.4985292071773069,
  'max_abs_difference': 581.302,
  'fraction_within_0_01': 0.9652991858351119},
 'MW_g_per_mol_vs_cleaned_rdkit_ExactMolWt': {'mean_abs_difference': 0.8398762097971916,
  'max_abs_difference': 582.543208352,
  'fraction_within_0_01': 7.904513477195479e-05}}

All imported source bytes are traced in provenance/import_manifest.json.
